# ЛР № 4. Витрина городского велопроката Сеула

**Основной вопрос:** Как различается спрос на аренду велосипедов по сезонам и часам суток?

Продолжаем [Project Brief](../docs/project_brief.md) и
[решения ЛР № 3](../docs/data_quality_report.md). В этом ноутбуке формируем почасовую витрину,
записываем её в S3 и проверяем результат для выбранного аналитического вопроса.

Срезы: Сезон аренды, час суток.

Метрики: Среднее число аренд по часу, доля аренд по сезону.

Модель и происхождение полей описаны в [docs/mart.md](../docs/mart.md).

## 1. Параметры и вход

Логическая дата берётся из пути подготовленного `staging`, а не из даты запуска.
Записи об аренде относятся к периоду `2017-12-01` – `2018-11-30`. `source_ingested_on=2026-10-15`
обозначает загрузку исходных данных и не используется как время события.

Ноутбук запускается из корня репозитория или каталога `notebooks` с ядром
проектного `.venv`. Стенд должен работать; настройки S3 читаются из `.env`.
Ingestion и ЛР № 3 повторно выполнять не нужно.

Справочник для выбранного вопроса не нужен. Читаем подготовленный Parquet напрямую
из `staging`; ingestion и проверки качества ЛР № 3 повторно не выполняем. На этом
шаге проверяем число строк и наличие полей для сезонного и часового срезов, меры
аренд и интерпретации нерабочих часов.


In [28]:
from datetime import date

DATASET_SLUG = "seoul_bike"
SOURCE_INGESTED_ON = "2026-10-15"
MART_NAME = "rent"
date.fromisoformat(SOURCE_INGESTED_ON)

PREFIX = f"{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}"
STAGING_PATH = f"s3://staging/{PREFIX}/staging.parquet"
MART_KEY = f"{PREFIX}/{MART_NAME}.parquet"
MART_PATH = f"s3://mart/{MART_KEY}"

print(f"Вход:  {STAGING_PATH}")
print(f"Выход: {MART_PATH}")

Вход:  s3://staging/seoul_bike/source_ingested_on=2026-10-15/staging.parquet
Выход: s3://mart/seoul_bike/source_ingested_on=2026-10-15/rent.parquet


In [29]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required_settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing_settings)}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402


def literal(value):
    return "'" + value.replace("'", "''") + "'"

def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

check_results = []

def check(label, actual, expected):
    passed = actual == expected
    check_results.append((label, actual, expected, passed))
    assert passed, f"{label}: получено {actual!r}, ожидалось {expected!r}"

connection = duckdb.connect()
configure_s3(connection)
endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
source_before = client.head_object(Bucket="staging", Key=f"{PREFIX}/staging.parquet")


In [ ]:
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging AS
    SELECT * FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false)
""")
source_schema = {row[0]: row[1] for row in connection.execute("DESCRIBE staging").fetchall()}
required_fields = {
    "Date", "Hour", "Seasons", "Rented Bike Count", "Functioning Day",
    "quality_functioning_day_consistency_violated",
}
check("Наличие нужных полей staging", required_fields - source_schema.keys(), set())

source_counts = connection.execute("""
    SELECT
        count(*) AS rows_count,
        count(*) FILTER (
            WHERE "Rented Bike Count" IS NULL
        ) AS null_rent_counts,
        count(*) FILTER (
            WHERE quality_functioning_day_consistency_violated
        ) AS flagged_rows,
        count(*) FILTER (
            WHERE "Date" IS NULL
               OR "Hour" IS NULL
               OR "Hour" NOT BETWEEN 0 AND 23
               OR "Seasons" IS NULL
               OR "Seasons" NOT IN ('Winter', 'Spring', 'Summer', 'Autumn')
               OR "Rented Bike Count" IS NULL
               OR "Rented Bike Count" < 0
               OR "Functioning Day" IS NULL
               OR quality_functioning_day_consistency_violated IS NULL
        ) AS critical_violations
    FROM staging
""").fetchone()
staging_rows, source_null_rent_counts, source_flagged_rows, source_critical = source_counts
check("Критические нарушения ЛР № 3 на входе", source_critical, 0)
check("Пропуски числа аренд", source_null_rent_counts, 0)

show_table(
    ["Строк staging", "Пропуски числа аренд", "Строк с флагом", "Критические нарушения"],
    [source_counts],
)
show_table(
    ["Нужное поле staging", "Тип"],
    [(name, source_schema[name]) for name in sorted(required_fields)],
)


rental_profile = connection.execute("""
    SELECT
        min("Rented Bike Count"),
        max("Rented Bike Count"),
        count(*) FILTER (WHERE NOT "Functioning Day"),
        count(*) FILTER (
            WHERE quality_functioning_day_consistency_violated
        )
    FROM staging
""").fetchone()

show_table(
    ["Минимум аренд", "Максимум аренд", "Нерабочих часов", "Часов с флагом"],
    [rental_profile],
)


Строк staging,Пропуски числа аренд,Строк с флагом,Критические нарушения
8760,0,0,0


Нужное поле staging,Тип
Date,DATE
Functioning Day,BOOLEAN
Hour,BIGINT
Rented Bike Count,BIGINT
Seasons,VARCHAR
quality_functioning_day_consistency_violated,BOOLEAN


Минимум аренд,Максимум аренд,Нерабочих часов,Часов с флагом
0,3556,295,0



Ограничение: при расчёте среднего спроса учитывать только рабочие часы.

## 2. Гранулярность, ключ и меры

**Одна строка таблицы `rent` представляет один часовой период наблюдения системы Seoul Bike Sharing
в определённую календарную дату** из выбранного снимка `staging`. 
Сезон и час суток — срезы этой часовой записи.

**Бизнес-ключ — `(Date, Hour)`.** Он идентифицирует час наблюдения системы. 
В ЛР № 3
проверены заполненность обеих частей и отсутствие повторов: 8 760 строк,
0 пропусков ключа и 0 повторов. Полных дубликатов также нет. В витрине не создаётся технический идентификатор.

Сохраняем все строки `staging`. Число строк (`count(*)`) означает число
часовых наблюдений, а не число аренд. Для основного вопроса используются
сезон (`Seasons`) и час (`Hour`), включая их сочетание.

| Мера | Единица и использование | Допустимая агрегация |
| --- | --- | --- |
| `Rented Bike Count` | Число арендованных велосипедов за час; основная мера спроса | Суммируется по непересекающимся часам для итога аренд. Среднее считается как `avg` по подходящим часовым строкам; средние групп нельзя складывать, а при объединении нужно учитывать число наблюдений. |
| Число строк (`count(*)`) | Число часовых наблюдений; знаменатель и контроль покрытия, не число аренд | Суммируется по непересекающимся группам часов. |

Для доли аренд сезона числитель — сумма `Rented Bike Count` по сезону,
знаменатель — сумма этой меры за весь анализируемый набор; долю нельзя
получить из количества строк. Сезонные суммы по непересекающимся сезонам
можно складывать. Витрина сама не хранит сезонные итоги или средние: они
рассчитываются последующей агрегацией часовых записей.

`Functioning Day = false` означает недоступность сервиса; соответствующие
нули аренд не трактуются как наблюдение отсутствия спроса при расчёте среднего.
Их нужно исключать из знаменателя среднего спроса либо показывать отдельно,
сохраняя при этом записи в витрине и учитывая их в покрытии. В ЛР № 3 таких
часов 295; нарушений согласованности и выставленных quality-флагов — 0.

Дата наблюдения (`Date`) относится к периоду `2017-12-01`–`2018-11-30`.
Логическая дата `2026-10-15` в пути — дата загрузки исходного снимка, а не
время события. Набор покрывает один полный год, поэтому сезонные сравнения
описывают только этот период и не гарантируют повторения в другие годы.


## 3. SQL-преобразование

Формируем широкую почасовую таблицу `rent` из полей, выбранных для вопроса о сезонном
и суточном профиле спроса. Явно перечисляем все поля, приводим имена к `snake_case`,
значения и уровень детализации не меняем. Сохраняем признак рабочего дня и флаг
проверки качества из ЛР № 3. Фильтрации, агрегации, соединений и дедупликации нет.

### Сборка широкой таблицы

In [31]:
connection.execute("""
    CREATE OR REPLACE TEMP TABLE rent AS
    SELECT
        "Date" AS date,
        "Hour" AS hour,
        "Seasons" AS seasons,
        "Rented Bike Count" AS rented_bike_count,
        "Functioning Day" AS functioning_day,
        quality_functioning_day_consistency_violated AS functioning_day_violation
    FROM staging
""")
prepared_rows = connection.execute("SELECT count(*) FROM rent").fetchone()[0]
excluded_rows = staging_rows - prepared_rows
prepared_flagged_rows = connection.execute("""
    SELECT count(*)
    FROM rent
    WHERE functioning_day_violation
""").fetchone()[0]

check("Число строк после преобразования", prepared_rows, staging_rows)
check("Исключённые строки", excluded_rows, 0)
check("Перенос quality-флага", prepared_flagged_rows, source_flagged_rows)

show_table(
    ["Поле витрины", "Тип"],
    [row[:2] for row in connection.execute("DESCRIBE rent").fetchall()],
)
show_table(
    ["Дата", "Час", "Сезон", "Аренды", "Рабочий день", "Нарушение-рабочего-дня"],
    connection.execute("""
        SELECT date, hour, seasons, rented_bike_count,
               functioning_day, functioning_day_violation
        FROM rent
        ORDER BY date, hour
        LIMIT 5
    """).fetchall(),
)

Поле витрины,Тип
date,DATE
hour,BIGINT
seasons,VARCHAR
rented_bike_count,BIGINT
functioning_day,BOOLEAN
functioning_day_violation,BOOLEAN


Дата,Час,Сезон,Аренды,Рабочий день,Нарушение-рабочего-дня
2017-12-01,0,Winter,254,True,False
2017-12-01,1,Winter,204,True,False
2017-12-01,2,Winter,173,True,False
2017-12-01,3,Winter,107,True,False
2017-12-01,4,Winter,78,True,False


## 4. Запись и повторное чтение S3

Записываем `rent` в Parquet по пути `MART_PATH`. Повторное выполнение заменяет
только целевой объект, не затрагивая другие объекты S3. Все строки `staging`
сохраняются, поэтому входное, подготовленное и повторно прочитанное количества
должны совпасть.

После записи открываем новое соединение DuckDB и читаем именно объект из бакета
`mart`; автоматическое добавление полей из каталогов отключено.

In [32]:
assert MART_PATH == f"s3://mart/{PREFIX}/rent.parquet"
assert MART_KEY == f"{PREFIX}/rent.parquet"
client.head_bucket(Bucket="mart")
client.delete_object(Bucket="mart", Key=MART_KEY)
connection.execute(f"""
    COPY rent TO {literal(MART_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")

published = duckdb.connect()
configure_s3(published)
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW published_rent AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")
mart_rows = published.execute("SELECT count(*) FROM published_rent").fetchone()[0]
mart_flagged_rows = published.execute("""
    SELECT count(*) FROM published_rent WHERE functioning_day_violation
""").fetchone()[0]
check("Количество строк после чтения из S3", mart_rows, staging_rows)
check("Флаги качества после чтения из S3", mart_flagged_rows, source_flagged_rows)

output_head = client.head_object(Bucket="mart", Key=MART_KEY)
show_table(
    ["Объект", "Строк", "Строк с quality-флагом", "Размер, байт"],
    [(MART_PATH, mart_rows, mart_flagged_rows, output_head["ContentLength"])],
)

Объект,Строк,Строк с quality-флагом,"Размер, байт"
s3://mart/seoul_bike/source_ingested_on=2026-10-15/rent.parquet,8760,0,32521


<a id="checks"></a>
## 5. Проверки сохранённой витрины

Ниже выполняются проверки для прочитанного из S3 Parquet-файла витрины и доступного в DuckDB как `published_rent`.
Проверки соответствуют выбранной почасовой модели: количество строк и схема,
составной ключ `(date, hour)`, обязательные поля и мера аренд, диапазоны срезов,
точный перенос quality-флага и контрольный срез `seasons × hour`.


### 1. Количество строк и схема

Сравниваем число записей `staging`, подготовленной таблицы и повторно прочитанного `mart`. Затем сверяем физические имена, порядок и типы полей файла с ожидаемой схемой.


In [33]:
# В представлении staging исходные имена полей приведены к именам витрины, для удобства.
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW source_staging AS
    SELECT
        "Date" AS date,
        "Hour" AS hour,
        "Seasons" AS seasons,
        "Rented Bike Count" AS rented_bike_count,
        "Functioning Day" AS functioning_day,
        quality_functioning_day_consistency_violated AS source_functioning_day_violation
    FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false)
""")
expected_schema = [
    ("date", "DATE"),
    ("hour", "BIGINT"),
    ("seasons", "VARCHAR"),
    ("rented_bike_count", "BIGINT"),
    ("functioning_day", "BOOLEAN"),
    ("functioning_day_violation", "BOOLEAN"),
]
actual_schema = [row[:2] for row in published.execute("DESCRIBE published_rent").fetchall()]
check("Имена, порядок и типы полей", actual_schema, expected_schema)
check("Баланс входных и сохранённых строк", staging_rows, mart_rows + excluded_rows)
check("Количество после чтения S3", mart_rows, prepared_rows)
show_table(["Staging", "Подготовлено", "Прочитано из mart", "Исключено"],
           [(staging_rows, prepared_rows, mart_rows, excluded_rows)])
show_table(["Поле", "Тип"], actual_schema)



Staging,Подготовлено,Прочитано из mart,Исключено
8760,8760,8760,0


Поле,Тип
date,DATE
hour,BIGINT
seasons,VARCHAR
rented_bike_count,BIGINT
functioning_day,BOOLEAN
functioning_day_violation,BOOLEAN


### 2. Ключ и обязательные поля

Проверяем пропуски в обязательных полях и бизнес-ключе `(date, hour)`, а также повторы ключа. Технический идентификатор не создаётся.


In [34]:
null_profile = published.execute("""
    SELECT
        count(*) FILTER (WHERE date IS NULL),
        count(*) FILTER (WHERE hour IS NULL),
        count(*) FILTER (WHERE seasons IS NULL),
        count(*) FILTER (WHERE rented_bike_count IS NULL),
        count(*) FILTER (WHERE functioning_day IS NULL),
        count(*) FILTER (WHERE functioning_day_violation IS NULL)
    FROM published_rent
""").fetchone()
for name, value in zip(("date", "hour", "seasons", "rented_bike_count",
                        "functioning_day", "functioning_day_violation"), null_profile):
    check(f"NULL в обязательном поле {name}", value, 0)
show_table(["Поле", "NULL", "Ожидается"],
           [(name, value, 0) for (name, _), value in zip(expected_schema, null_profile)])

key_profile = published.execute("""
    SELECT
        count(*) FILTER (WHERE date IS NULL OR hour IS NULL),
        COALESCE((SELECT sum(rows_per_key - 1) FROM (
            SELECT count(*) AS rows_per_key
            FROM published_rent
            GROUP BY date, hour HAVING count(*) > 1
        )), 0)
    FROM published_rent
""").fetchone()
check("Пропуски составного ключа (date, hour)", key_profile[0], 0)
check("Повторы составного ключа (date, hour)", key_profile[1], 0)
show_table(["Проверка ключа", "Нарушений"], [
    ("NULL в date или hour", key_profile[0]),
    ("Повторяющиеся пары date, hour", key_profile[1]),
])



Поле,NULL,Ожидается
date,0,0
hour,0,0
seasons,0,0
rented_bike_count,0,0
functioning_day,0,0
functioning_day_violation,0,0


Проверка ключа,Нарушений
NULL в date или hour,0
"Повторяющиеся пары date, hour",0


### 3. Мера и диапазоны

Показываем общий диапазон числа аренд и отдельно диапазон за рабочие часы, поскольку нули в нерабочие часы отражают недоступность сервиса. Проверки отрицательных значений, часа и сезона выполняются по всем строкам.


In [35]:
measure_profile = published.execute("""
    SELECT min(rented_bike_count), max(rented_bike_count),
           min(rented_bike_count) FILTER (WHERE functioning_day),
           max(rented_bike_count) FILTER (WHERE functioning_day),
           count(*) FILTER (WHERE rented_bike_count < 0),
           count(*) FILTER (WHERE hour NOT BETWEEN 0 AND 23),
           count(*) FILTER (WHERE seasons NOT IN ('Winter', 'Spring', 'Summer', 'Autumn'))
    FROM published_rent
""").fetchone()
check("Отрицательные значения числа аренд", measure_profile[4], 0)
check("Час вне диапазона 0-23", measure_profile[5], 0)
check("Сезон вне справочника", measure_profile[6], 0)
show_table(["Минимум аренд (все часы)", "Максимум аренд (все часы)",
            "Минимум аренд (рабочие часы)", "Максимум аренд (рабочие часы)",
            "Отрицательные", "Неверный час", "Неверный сезон"],
           [measure_profile])



Минимум аренд (все часы),Максимум аренд (все часы),Минимум аренд (рабочие часы),Максимум аренд (рабочие часы),Отрицательные,Неверный час,Неверный сезон
0,3556,2,3556,0,0,0


### 4. Флаги и сохранение значений

Сверяем число и точный перенос quality-флага со `staging`. `EXCEPT ALL` в обе стороны проверяет все значения и кратность строк, не скрывая совпадающие строки.


In [36]:
flag_profile = published.execute("""
    SELECT
        count(*) FILTER (WHERE functioning_day_violation) AS mart_flagged,
        count(*) FILTER (WHERE source_functioning_day_violation) AS staging_flagged,
        count(*) FILTER (
            WHERE functioning_day_violation IS DISTINCT FROM source_functioning_day_violation
        ) AS mismatches
    FROM published_rent p
    JOIN source_staging s ON p.date = s.date AND p.hour = s.hour
""").fetchone()
check("Количество quality-флагов относительно staging", flag_profile[0], source_flagged_rows)
check("Точный перенос quality-флага", flag_profile[2], 0)
show_table(["Флаги в mart", "Флаги в staging", "Несоответствия"], [flag_profile])



Флаги в mart,Флаги в staging,Несоответствия
0,0,0


### 5. Пригодность для выбранного вопроса

Группировка по `seasons` и `hour` подтверждает, что сохранённая витрина содержит
оба необходимых среза и их сочетание. Сумма количества строк по группам должна
совпасть с числом строк витрины; содержательные выводы и расчёт средних относятся
к следующему этапу.


In [37]:
season_hour = published.execute("""
    SELECT seasons, hour, count(*) AS observations,
           sum(rented_bike_count) AS rentals_total,
           avg(rented_bike_count) FILTER (WHERE functioning_day) AS rentals_avg_working
    FROM published_rent
    GROUP BY seasons, hour
    ORDER BY seasons, hour
""").fetchall()
check("Баланс группировки season × hour", sum(row[2] for row in season_hour), mart_rows)
check("Число комбинаций season × hour", len(season_hour), 96)
show_table(["Сезон", "Час", "Наблюдений", "Сумма аренд", "Среднее в рабочие часы"],
           season_hour[:12])
show_table(["Всего групп", "Сумма наблюдений по группам"],
           [(len(season_hour), sum(row[2] for row in season_hour))])
show_table(["Всего проверок", "Пройдено", "Строк mart"], [
    (len(check_results), sum(passed for _, _, _, passed in check_results), mart_rows)
])


Сезон,Час,Наблюдений,Сумма аренд,Среднее в рабочие часы
Autumn,0,91,56755,709.4375
Autumn,1,91,44200,552.5
Autumn,2,91,30198,377.475
Autumn,3,91,20524,256.55
Autumn,4,91,13522,169.025
Autumn,5,91,13073,163.4125
Autumn,6,91,28759,359.4875
Autumn,7,91,63899,788.8765432098766
Autumn,8,91,108948,1345.037037037037
Autumn,9,91,68723,848.4320987654321


Всего групп,Сумма наблюдений по группам
96,8760


Всего проверок,Пройдено,Строк mart
26,26,8760


## 6. Оформление результата

Модель и происхождение полей описаны в [docs/mart.md](../docs/mart.md).
Витрина содержит одну широкую почасовую таблицу для вопроса о сезонном и суточном
профиле спроса.

Итог проверок: все 8 760 строк `staging` сохранены в `rent`, исключённых строк нет;
составной ключ `(date, hour)` уникален, обязательные поля заполнены, часы и сезоны
находятся в допустимых диапазонах, а quality-флаг перенесён без изменений.
Контрольная группировка `seasons × hour` содержит 96 комбинаций и сохраняет баланс
наблюдений. 295 нерабочих часов остаются в витрине, но исключаются из знаменателя
среднего спроса согласно ограничению ЛР № 3.

Витрина сохранена и прочитана из S3:

```text
s3://mart/seoul_bike/source_ingested_on=2026-10-15/rent.parquet
```

Ограничения: данные описывают один год и фактически совершённые аренды одного города.
Нерабочий час означает недоступность сервиса, а не нулевой спрос. Полный расчёт
метрик и содержательные выводы выполняются на следующем этапе.
